In [1]:
#### in this script, calculate rolling averages/merging model and observations ####

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import boto3
import datetime

In [2]:
nbm_s3_url = "s3://processed-data-809918852303-us-east-1-an/nbm_maxt_DEPLOYMENT.parquet"
nbm_df = pd.read_parquet(nbm_s3_url)

In [3]:
nbm_df['date'] = pd.to_datetime(nbm_df['date'])
nbm_df = nbm_df.sort_values(by=['public_zone', 'date', 'week', 'forecast_day'])

In [4]:
prism_s3_url = 's3://processed-data-809918852303-us-east-1-an/prism_maxt_DEPLOYMENT.parquet'
prism_df = pd.read_parquet(prism_s3_url)
prism_df = prism_df.sort_values(["unique_zone_str", "date"])

In [6]:
# shift by 1 day to exclude the current day!!
prism_df["60daywindow_prism"] = prism_df.groupby("unique_zone_str")["maxt_value"]
    .transform(lambda x: x.shift(1).rolling(window=60, min_periods=1).mean())

prism_df["7daywindow_prism"] = prism_df.groupby("unique_zone_str")["maxt_value"].
    transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())

In [7]:
# forward fill future dates to create full date spine for observations per zone up to the max forecast date in df_model
# so NaNs don't propagate
max_forecast_date = nbm_df["date"].max()
zones = prism_df["unique_zone_str"].unique()

full_idx = pd.MultiIndex.from_product(
    [zones, pd.date_range(prism_df["date"].min(), max_forecast_date)],
    names=["unique_zone_str", "date"],
)

In [8]:
prism_df_extended = (
    prism_df.set_index(["unique_zone_str", "date"])
    .reindex(full_idx)
    .groupby("unique_zone_str")
    .ffill()  # Forward-fills latest valid rolling window metrics into future dates, 6 days in advance
    .reset_index()
)

In [9]:
prism_df_extended

,unique_zone_str,date,global_zone_id,maxt_value,week,zone_id,state,name,60daywindow_prism,7daywindow_prism
0,AL_001,2026-07-06,1.0,88.467347,28,001,AL,Lauderdale,NaN,NaN
1,AL_001,2026-07-07,1.0,88.673932,28,001,AL,Lauderdale,88.467347,88.467347
2,AL_001,2026-07-08,1.0,88.884477,28,001,AL,Lauderdale,88.570639,88.570639
3,AL_001,2026-07-09,1.0,90.631315,28,001,AL,Lauderdale,88.675252,88.675252
4,AL_001,2026-07-10,1.0,87.198984,28,001,AL,Lauderdale,89.164268,89.164268
...,...,...,...,...,...,...,...,...,...,...
265645,WY_199,2026-09-08,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
265646,WY_199,2026-09-09,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
265647,WY_199,2026-09-10,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
265648,WY_199,2026-09-11,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780


In [15]:
merged_df = pd.merge(
    nbm_df,
    prism_df_extended,
    on=["date", "unique_zone_str"],
    suffixes=("_nbm", "_prism"),
    how="left",
)

In [16]:
merged_df

,public_zone,date,week_nbm,maxt_value_nbm,forecast_day,unique_zone_str,model_run_date,zone_id_nbm,state_nbm,name_nbm,global_zone_id,maxt_value_prism,week_prism,zone_id_prism,state_prism,name_prism,60daywindow_prism,7daywindow_prism
0,1,2026-07-07,28,88.273973,1,AL_001,2026-07-07,001,AL,Lauderdale,1.0,88.673932,28,001,AL,Lauderdale,88.467347,88.467347
1,1,2026-07-08,28,88.609589,1,AL_001,2026-07-08,001,AL,Lauderdale,1.0,88.884477,28,001,AL,Lauderdale,88.570639,88.570639
2,1,2026-07-08,28,88.191781,2,AL_001,2026-07-07,001,AL,Lauderdale,1.0,88.884477,28,001,AL,Lauderdale,88.570639,88.570639
3,1,2026-07-09,28,89.791096,1,AL_001,2026-07-09,001,AL,Lauderdale,1.0,90.631315,28,001,AL,Lauderdale,88.675252,88.675252
4,1,2026-07-09,28,90.510274,2,AL_001,2026-07-08,001,AL,Lauderdale,1.0,90.631315,28,001,AL,Lauderdale,88.675252,88.675252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667045,3850,2026-09-10,37,83.655758,6,WY_199,2026-09-05,199,WY,Sheridan Foothills,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
1667046,3850,2026-09-10,37,81.772121,7,WY_199,2026-09-04,199,WY,Sheridan Foothills,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
1667047,3850,2026-09-11,37,81.018182,6,WY_199,2026-09-06,199,WY,Sheridan Foothills,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780
1667048,3850,2026-09-11,37,81.004848,7,WY_199,2026-09-05,199,WY,Sheridan Foothills,3850.0,98.672012,36,199,WY,Sheridan Foothills,89.320974,84.735780


In [17]:
merged_df = merged_df.drop(columns=['week_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 'global_zone_id', 'unique_zone_str'])
merged_df = merged_df.rename(columns={'week_prism': 'week', 'state_prism': 'state', 'name_prism':'name', 'zone_id_prism':'zone_id'})

In [18]:
merged_df['nbm_minus_obs'] = merged_df['maxt_value_nbm'] - merged_df['maxt_value_prism']

In [19]:
column_order = [
        'public_zone', 'date', 'week', 'forecast_day', 'state', 'zone_id', 'name', 
        'model_run_date', 'maxt_value_nbm', 'maxt_value_prism', '60daywindow_prism', '7daywindow_prism','nbm_minus_obs'
]
merged_df = merged_df[column_order]

In [20]:
merged_df

,public_zone,date,week,forecast_day,state,zone_id,name,model_run_date,maxt_value_nbm,maxt_value_prism,60daywindow_prism,7daywindow_prism,nbm_minus_obs
0,1,2026-07-07,28,1,AL,001,Lauderdale,2026-07-07,88.273973,88.673932,88.467347,88.467347,-0.399960
1,1,2026-07-08,28,1,AL,001,Lauderdale,2026-07-08,88.609589,88.884477,88.570639,88.570639,-0.274888
2,1,2026-07-08,28,2,AL,001,Lauderdale,2026-07-07,88.191781,88.884477,88.570639,88.570639,-0.692696
3,1,2026-07-09,28,1,AL,001,Lauderdale,2026-07-09,89.791096,90.631315,88.675252,88.675252,-0.840219
4,1,2026-07-09,28,2,AL,001,Lauderdale,2026-07-08,90.510274,90.631315,88.675252,88.675252,-0.121041
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667045,3850,2026-09-10,36,6,WY,199,Sheridan Foothills,2026-09-05,83.655758,98.672012,89.320974,84.735780,-15.016255
1667046,3850,2026-09-10,36,7,WY,199,Sheridan Foothills,2026-09-04,81.772121,98.672012,89.320974,84.735780,-16.899891
1667047,3850,2026-09-11,36,6,WY,199,Sheridan Foothills,2026-09-06,81.018182,98.672012,89.320974,84.735780,-17.653831
1667048,3850,2026-09-11,36,7,WY,199,Sheridan Foothills,2026-09-05,81.004848,98.672012,89.320974,84.735780,-17.667164


In [21]:
merged_df['nbm_minus_obs_window_60d'] = merged_df['maxt_value_nbm'] - merged_df['60daywindow_prism']
merged_df['nbm_minus_obs_window_7d'] = merged_df['maxt_value_nbm'] - merged_df['7daywindow_prism']

In [22]:
merged_df

,public_zone,date,week,forecast_day,state,zone_id,name,model_run_date,maxt_value_nbm,maxt_value_prism,60daywindow_prism,7daywindow_prism,nbm_minus_obs,nbm_minus_obs_window_60d,nbm_minus_obs_window_7d
0,1,2026-07-07,28,1,AL,001,Lauderdale,2026-07-07,88.273973,88.673932,88.467347,88.467347,-0.399960,-0.193374,-0.193374
1,1,2026-07-08,28,1,AL,001,Lauderdale,2026-07-08,88.609589,88.884477,88.570639,88.570639,-0.274888,0.038950,0.038950
2,1,2026-07-08,28,2,AL,001,Lauderdale,2026-07-07,88.191781,88.884477,88.570639,88.570639,-0.692696,-0.378859,-0.378859
3,1,2026-07-09,28,1,AL,001,Lauderdale,2026-07-09,89.791096,90.631315,88.675252,88.675252,-0.840219,1.115844,1.115844
4,1,2026-07-09,28,2,AL,001,Lauderdale,2026-07-08,90.510274,90.631315,88.675252,88.675252,-0.121041,1.835022,1.835022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667045,3850,2026-09-10,36,6,WY,199,Sheridan Foothills,2026-09-05,83.655758,98.672012,89.320974,84.735780,-15.016255,-5.665216,-1.080023
1667046,3850,2026-09-10,36,7,WY,199,Sheridan Foothills,2026-09-04,81.772121,98.672012,89.320974,84.735780,-16.899891,-7.548852,-2.963659
1667047,3850,2026-09-11,36,6,WY,199,Sheridan Foothills,2026-09-06,81.018182,98.672012,89.320974,84.735780,-17.653831,-8.302792,-3.717598
1667048,3850,2026-09-11,36,7,WY,199,Sheridan Foothills,2026-09-05,81.004848,98.672012,89.320974,84.735780,-17.667164,-8.316125,-3.730932


In [23]:
print(merged_df['nbm_minus_obs'].describe().round(2))
print(merged_df['nbm_minus_obs_window_60d'].describe().round(2))
print(merged_df['nbm_minus_obs_window_7d'].describe().round(2))

count    1667050.00
mean          -0.32
std            3.70
min          -37.30
25%           -2.17
50%           -0.42
75%            1.48
max           28.28
Name: nbm_minus_obs, dtype: float64
count    1667050.00
mean          -0.19
std            4.79
min          -32.89
25%           -2.84
50%            0.07
75%            2.84
max           25.20
Name: nbm_minus_obs_window_60d, dtype: float64
count    1667050.00
mean          -0.07
std            4.45
min          -27.65
25%           -2.71
50%            0.06
75%            2.65
max           25.20
Name: nbm_minus_obs_window_7d, dtype: float64


In [24]:
BUCKET_NAME = 'processed-data-809918852303-us-east-1-an'

merged_df.to_parquet('merged_maxt_DEPLOYMENT.parquet', index=False)
s3 = boto3.client('s3')
s3.upload_file('merged_maxt_DEPLOYMENT.parquet', BUCKET_NAME, 'merged_maxt_DEPLOYMENT.parquet')
print("Parquet complete /uploaded  now")

Parquet complete /uploaded  now


In [ ]:
# attempting to plot NBM biases

plot_day = '2026-09-06'
day_diff_df = merged_df[merged_df['date'] == plot_day]

zones_gdf = gpd.read_file("https://www.weather.gov/source/gis/Shapefiles/WSOM/z_16ap26.zip")
zones_gdf['unique_zone_str'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']
zones_gdf['STATE_ZONE'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']

zones_display = zones_gdf.to_crs(epsg=4269)
map_gdf = zones_display.merge(day_diff_df, left_on='STATE_ZONE', right_on='unique_zone_str')

#initialize the plot figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
zones_display.plot(ax=ax, color='lightgrey', edgecolor='none')
plt.grid(linestyle='--', color='grey', alpha=0.2)

plt.xlim(-126, -65)
plt.ylim(24, 50)

max_error_bound = 10

map_gdf.plot(
    column='temp_diff_f',
    ax=ax,
    legend=True,
    legend_kwds={'label': 'temperature diff in °F', 'orientation': 'horizontal', 'extend': 'both','pad': 0.05},
    cmap='RdBu_r',
    vmin=-max_error_bound,
    vmax=max_error_bound
)

plt.title('NBM bias for ' + plot_day + ' forecast day 1', fontsize=13, fontweight='bold') # this is model - obs
plt.tight_layout()
plt.show()